In [1]:
!pip install torch torchvision
!pip install transformers==4.30.2
!pip install numpy==1.24.4
!pip install scikit-learn==1.2.2
!pip install git+https://github.com/RyanWangZf/MedCLIP.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [2]:
import json
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
import torch.nn as nn

class ENTMedClipDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, metadata_path):
        self.img_dir = img_dir
        self.items = []
        with open(metadata_path) as f:
            for line in f:
                self.items.append(json.loads(line))

    def __getitem__(self, idx):
        item = self.items[idx]
        img_path = os.path.join(self.img_dir, "images", item["image"])
        image = Image.open(img_path).convert("RGB")
        text  = item["text"]
        return {"image": image, "text": text}

    def __len__(self):
        return len(self.items)

In [3]:
!pip install protobuf==3.20.3 --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-me

In [4]:
import torch
from medclip import MedCLIPModel, MedCLIPVisionModelViT, MedCLIPProcessor

class MedCLIPForENT(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = MedCLIPModel()

    def forward(
        self,
        pixel_values=None,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        **kwargs,
    ):
        outputs = self.model(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # ONLY logits exist in this MedCLIP version
        return {
            "logits": outputs["logits"]
        }

2025-11-28 07:47:52.243154: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764316072.447655      21 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764316072.504225      21 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [5]:
from medclip import MedCLIPProcessor
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import os
import torch.nn.functional as F
import torch

processor = MedCLIPProcessor()

train_ds = ENTMedClipDataset(
    "/kaggle/input/entrep/entrep_processed/train",
    "/kaggle/input/entrep/entrep_processed/train/metadata_train.jsonl"
)
val_ds = ENTMedClipDataset(
    "/kaggle/input/entrep/entrep_processed/val",
    "/kaggle/input/entrep/entrep_processed/val/metadata_val.jsonl"
)

def collate_fn(batch):
    images = [x["image"] for x in batch]
    texts  = [x["text"]  for x in batch]
    return {"image": images, "text": texts}

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=8, shuffle=False, collate_fn=collate_fn)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MedCLIPForENT().to(device)

optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-3)

# =============================
# ⭐ Thêm biến lưu best model
# =============================
best_val_loss = float("inf")
best_path = "/kaggle/working/medclip_best.pth"

# =============================
# ⭐ Training Loop
# =============================
for epoch in range(15):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        inputs = processor(
            images=batch["image"],
            text=batch["text"],
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        outputs = model(**inputs)
        logits = outputs["logits"]
        
        labels = torch.arange(logits.size(0)).to(device)
        
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        loss = (loss_i2t + loss_t2i) / 2

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"[Epoch {epoch+1}] Train Loss = {avg_train_loss:.4f}")

    # =============================
    # ⭐ Validation
    # =============================
    model.eval()
    val_loss_sum = 0

    with torch.no_grad():
        for batch in val_loader:
            inputs = processor(
                images=batch["image"],
                text=batch["text"],
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(device)

            outputs = model(**inputs)
            logits = outputs["logits"]
            labels = torch.arange(logits.size(0)).to(device)

            loss = (
                F.cross_entropy(logits, labels) +
                F.cross_entropy(logits.T, labels)
            ) / 2

            val_loss_sum += loss.item()

    avg_val_loss = val_loss_sum / len(val_loader)
    print(f"Val Loss = {avg_val_loss:.4f}")

    # =============================
    # ⭐ Lưu BEST MODEL
    # =============================
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), best_path)
        print(f"🔥 Saved BEST model (Val Loss = {avg_val_loss:.4f}) → {best_path}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 181MB/s]
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when 

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

...

[Epoch 13] Train Loss = 0.2005
Val Loss = 1.2069


100%|██████████| 57/57 [00:12<00:00,  4.54it/s]


[Epoch 14] Train Loss = 0.1762
Val Loss = 1.2346


100%|██████████| 57/57 [00:12<00:00,  4.51it/s]


[Epoch 15] Train Loss = 0.1781
Val Loss = 1.2696


In [6]:
# test_retrieval.py
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

# ====== 1. Load Processor & Model ======
processor = MedCLIPProcessor()
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MedCLIPForENT().to(device)
model.load_state_dict(torch.load("/kaggle/working/medclip_best.pth", map_location=device))
model.eval()

# ====== 2. Load Test Dataset ======
test_ds = ENTMedClipDataset(
    "/kaggle/input/entrep/entrep_processed/test",
    "/kaggle/input/entrep/entrep_processed/test/metadata_test.jsonl"
)
test_loader = DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

# ====== 3. Storage ======
image_embeds = []
text_embeds = []

# ====== 4. Extract Embeddings ======
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Extracting embeddings"):
        
        images = batch["image"]  # list[PIL]
        texts  = batch["text"]   # list[str]

        # ---- Processor ----
        inputs = processor(
            images=images,
            text=texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        # ---- Forward USING MEDCLIPMODEL directly ----
        # (Not using model(**inputs), because MedCLIPForENT drops embeddings)
        outputs = model.model(**inputs)

        img_feat  = outputs["img_embeds"]     # (1, D)
        text_feat = outputs["text_embeds"]    # (1, D)

        # ---- Normalize ----
        img_feat  = img_feat / img_feat.norm(dim=-1, keepdim=True)
        text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)

        # ---- Collect ----
        image_embeds.append(img_feat.cpu().numpy())
        text_embeds.append(text_feat.cpu().numpy())

# ====== 5. Final Arrays ======
image_embeds = np.vstack(image_embeds)
text_embeds  = np.vstack(text_embeds)

print("Image embeddings shape:", image_embeds.shape)
print("Text embeddings shape :", text_embeds.shape)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Some weights of the model checkpoint at emilyalsentzer/Bio_ClinicalBERT were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weig

Image embeddings shape: (58, 512)
Text embeddings shape : (58, 512)


In [7]:
import numpy as np

# =========================================================
# Recall@K
# =========================================================
def recall_at_k(sim_matrix, k):
    N = sim_matrix.shape[0]
    k = min(k, N)

    ranking = np.argsort(-sim_matrix, axis=1)
    hits = np.array([1 if i in ranking[i, :k] else 0 for i in range(N)])

    return hits.mean()


# =========================================================
# Precision@K
# =========================================================
def precision_at_k(sim_matrix, k):
    N = sim_matrix.shape[0]
    k = min(k, N)

    ranking = np.argsort(-sim_matrix, axis=1)
    precisions = np.array([
        (1 if i in ranking[i, :k] else 0) / k
        for i in range(N)
    ])

    return precisions.mean()


# =========================================================
# Average Precision cho 1 sample
# =========================================================
def average_precision(ranking, gt_index):
    score = 0.0
    correct = 0
    for rank, idx in enumerate(ranking, start=1):
        if idx == gt_index:
            correct += 1
            score += correct / rank
    return score


# =========================================================
# Mean Average Precision (mAP)
# =========================================================
def mean_average_precision(sim_matrix):
    N = sim_matrix.shape[0]
    ranking = np.argsort(-sim_matrix, axis=1)

    aps = np.array([
        average_precision(ranking[i], i)
        for i in range(N)
    ])
    return aps.mean()


# =========================================================
# nDCG@K
# =========================================================
def ndcg_at_k(sim_matrix, k):
    N = sim_matrix.shape[0]
    k = min(k, N)

    ranking = np.argsort(-sim_matrix, axis=1)
    ndcgs = []

    for i in range(N):
        topk = ranking[i, :k]
        rel = np.array([1 if idx == i else 0 for idx in topk])

        dcg = np.sum(rel / np.log2(np.arange(2, k+2)))
        idcg = 1.0  # vì chỉ có 1 ground truth
        ndcgs.append(dcg / idcg)

    return np.mean(ndcgs)


In [8]:
sim_matrix = image_embeds @ text_embeds.T

print("Recall@1 :", recall_at_k(sim_matrix, 1))
print("Recall@5 :", recall_at_k(sim_matrix, 5))
print("Recall@10:", recall_at_k(sim_matrix, 10))

print("Precision@1:", precision_at_k(sim_matrix, 1))
print("Precision@5:", precision_at_k(sim_matrix, 5))
print("Precision@10:", precision_at_k(sim_matrix, 10))


print("mAP:", mean_average_precision(sim_matrix))

print("nDCG@1:", ndcg_at_k(sim_matrix, 1))
print("nDCG@5:", ndcg_at_k(sim_matrix, 5))
print("nDCG@10:", ndcg_at_k(sim_matrix, 10))


Recall@1 : 0.15517241379310345
Recall@5 : 0.4482758620689655
Recall@10: 0.7758620689655172
Precision@1: 0.15517241379310345
Precision@5: 0.0896551724137931
Precision@10: 0.07758620689655173
mAP: 0.2991911106135243
nDCG@1: 0.15517241379310345
nDCG@5: 0.2920105575243999
nDCG@10: 0.3965274126792658


In [9]:
import numpy as np

def compute_mrr(sim_matrix):
    N = sim_matrix.shape[0]
    rr = []
    for i in range(N):
        ranks = np.argsort(-sim_matrix[i])
        pos = np.where(ranks == i)[0][0]
        rr.append(1 / (pos + 1))
    return np.mean(rr)

print("MRR (text → image):", compute_mrr(sim_matrix))

MRR (text → image): 0.2991911106135243


In [10]:
def get_topk_candidates(sim_matrix, query_idx, k=20):
    scores = sim_matrix[query_idx]          # shape (N,)
    topk_idx = np.argsort(scores)[::-1][:k] # descending
    return topk_idx, scores[topk_idx]

from openai import OpenAI
client = OpenAI()

def llm_score(query, caption):
    prompt = f"""
You are an ENT medical specialist.
Query: "{query}"
Image Description: "{caption}"

Score how relevant this image is to the query (0 to 1).
Respond with ONLY a number between 0 and 1.
"""

    out = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    text = out.choices[0].message["content"].strip()
    return float(text)


OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable